#### Understand the JSON Structure

In [42]:
import json
import pandas as pd
import os
from tqdm import tqdm

In [43]:
data_folder = "E:\Data analysis projects\cricket_odi_analytics\data\odis_json"

In [44]:
files = os.listdir(data_folder)

print("Total files:", len(files))
print("Example file:", files[0])

Total files: 3099
Example file: 1000887.json


#### Open One Match File

In [45]:
sample_file = os.path.join(data_folder, files[0])

with open(sample_file) as f:
    data = json.load(f)

data.keys()

dict_keys(['meta', 'info', 'innings'])

#### Inspect Innings Structure

In [46]:
data["innings"][0].keys()

dict_keys(['team', 'overs', 'powerplays'])

In [47]:
data["innings"][0]["overs"][0]

{'over': 0,
 'deliveries': [{'batter': 'DA Warner',
   'bowler': 'Mohammad Amir',
   'non_striker': 'TM Head',
   'runs': {'batter': 0, 'extras': 0, 'total': 0}},
  {'batter': 'DA Warner',
   'bowler': 'Mohammad Amir',
   'non_striker': 'TM Head',
   'runs': {'batter': 0, 'extras': 0, 'total': 0}},
  {'batter': 'DA Warner',
   'bowler': 'Mohammad Amir',
   'non_striker': 'TM Head',
   'runs': {'batter': 0, 'extras': 0, 'total': 0}},
  {'batter': 'DA Warner',
   'bowler': 'Mohammad Amir',
   'non_striker': 'TM Head',
   'runs': {'batter': 0, 'extras': 0, 'total': 0}},
  {'batter': 'DA Warner',
   'bowler': 'Mohammad Amir',
   'extras': {'wides': 1},
   'non_striker': 'TM Head',
   'runs': {'batter': 0, 'extras': 1, 'total': 1}},
  {'batter': 'DA Warner',
   'bowler': 'Mohammad Amir',
   'non_striker': 'TM Head',
   'runs': {'batter': 0, 'extras': 0, 'total': 0}},
  {'batter': 'DA Warner',
   'bowler': 'Mohammad Amir',
   'non_striker': 'TM Head',
   'runs': {'batter': 0, 'extras': 0, 't

### Inspect One Delivery

In [48]:
data["innings"][0]["overs"][0]["deliveries"][0]

{'batter': 'DA Warner',
 'bowler': 'Mohammad Amir',
 'non_striker': 'TM Head',
 'runs': {'batter': 0, 'extras': 0, 'total': 0}}

### Build the Ball-by-Ball Extraction Script

#### Create Storage Lists

In [49]:
all_deliveries = []
bad_files = []


#### Main Extraction Loop

In [50]:
for file in tqdm(files):
    file_path = os.path.join(data_folder, file)

    try:
        with open(file_path) as f:
            data = json.load(f)
    except:
        bad_files.append(file)
        continue

    match_id = file.replace(".json", "")
    
    # Extract match date
    match_date = None
    if "info" in data and "dates" in data["info"]:
        # dates is a list, usually with one date, but could have multiple for multi-day matches
        match_date = data["info"]["dates"][0] if data["info"]["dates"] else None

    for inning_no, inning in enumerate(data["innings"], start=1):

        team = inning["team"]

        for over in inning["overs"]:
            over_no = over["over"]

            for ball_no, delivery in enumerate(over["deliveries"], start=1):

                batter = delivery.get("batter")
                bowler = delivery.get("bowler")
                non_striker = delivery.get("non_striker")

                runs_batter = delivery["runs"]["batter"]
                runs_total = delivery["runs"]["total"]
                extras = delivery["runs"].get("extras", 0)
                
                # Check if wicket occurred
                wicket = 1 if "wickets" in delivery else 0
                
                # Get wicket details
                player_out = None
                wicket_kind = None
                fielder_names = None
                
                if "wickets" in delivery:
                    wicket_info = delivery["wickets"][0]  # Usually only one wicket per ball
                    player_out = wicket_info.get("player_out")
                    wicket_kind = wicket_info.get("kind")
                    
                    # Extract fielder names
                    if "fielders" in wicket_info:
                        fielders = wicket_info["fielders"]
                        # Handle both list of dicts and other formats
                        if isinstance(fielders, list):
                            fielder_names = ", ".join([f.get("name", "") for f in fielders if f.get("name")])
                        elif isinstance(fielders, dict):
                            fielder_names = fielders.get("name")
                        else:
                            fielder_names = str(fielders)

                all_deliveries.append({
                    "match_id": match_id,
                    "match_date": match_date,  # New column for match date
                    "inning": inning_no,
                    "over": over_no,
                    "ball": ball_no,
                    "batter": batter,
                    "bowler": bowler,
                    "non_striker": non_striker,
                    "runs_batter": runs_batter,
                    "runs_total": runs_total,
                    "extras": extras,
                    "wicket": wicket,
                    "player_out": player_out,  
                    "wicket_kind": wicket_kind,
                    "fielder": fielder_names,  # New column for fielder name(s)
                    "team": team  
                })

  0%|          | 0/3099 [00:00<?, ?it/s]

100%|██████████| 3099/3099 [02:09<00:00, 23.89it/s]


In [51]:
len(bad_files)

1

#### Convert to DataFrame

In [52]:
df = pd.DataFrame(all_deliveries)

### Data Cleaning & Data Validation

#### Inspect Dataset

In [53]:
df.shape

(1639277, 16)

In [54]:
df.head()

,match_id,match_date,inning,over,ball,batter,bowler,non_striker,runs_batter,runs_total,extras,wicket,player_out,wicket_kind,fielder,team
0,1000887,2017-01-13,1,0,1,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia
1,1000887,2017-01-13,1,0,2,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia
2,1000887,2017-01-13,1,0,3,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia
3,1000887,2017-01-13,1,0,4,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia
4,1000887,2017-01-13,1,0,5,DA Warner,Mohammad Amir,TM Head,0,1,1,0,None,None,None,Australia


In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1639277 entries, 0 to 1639276
Data columns (total 16 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   match_id     1639277 non-null  object
 1   match_date   1639277 non-null  object
 2   inning       1639277 non-null  int64 
 3   over         1639277 non-null  int64 
 4   ball         1639277 non-null  int64 
 5   batter       1639277 non-null  object
 6   bowler       1639277 non-null  object
 7   non_striker  1639277 non-null  object
 8   runs_batter  1639277 non-null  int64 
 9   runs_total   1639277 non-null  int64 
 10  extras       1639277 non-null  int64 
 11  wicket       1639277 non-null  int64 
 12  player_out   44877 non-null    object
 13  wicket_kind  44877 non-null    object
 14  fielder      28934 non-null    object
 15  team         1639277 non-null  object
dtypes: int64(7), object(9)
memory usage: 200.1+ MB


In [68]:
df['team'].unique()

array(['Australia', 'Pakistan', 'New Zealand', 'Scotland', 'Hong Kong',
       'Zimbabwe', 'India', 'Bangladesh', 'South Africa', 'England',
       'Sri Lanka', 'Papua New Guinea', 'West Indies', 'Ireland',
       'United Arab Emirates', 'Nepal', 'United States of America',
       'Namibia', 'Oman', 'Netherlands', 'Thailand', 'Canada', 'Jersey',
       'Africa XI', 'Asia XI', 'ICC World XI', 'Bermuda', 'Kenya'],
      dtype=object)

#### Check Missing Values

In [56]:
df.isnull().sum()

match_id             0
match_date           0
inning               0
over                 0
ball                 0
batter               0
bowler               0
non_striker          0
runs_batter          0
runs_total           0
extras               0
wicket               0
player_out     1594400
wicket_kind    1594400
fielder        1610343
team                 0
dtype: int64

#### Basic Statistics

In [57]:
df.describe()

,inning,over,ball,runs_batter,runs_total,extras,wicket
count,1.639277e+06,1.639277e+06,1.639277e+06,1.639277e+06,1.639277e+06,1.639277e+06,1.639277e+06
mean,1.455634e+00,2.218744e+01,3.594477e+00,7.786018e-01,8.277509e-01,4.914911e-02,2.737609e-02
std,4.982998e-01,1.379123e+01,1.790558e+00,1.247702e+00,1.254493e+00,2.951686e-01,1.631768e-01
min,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,1.000000e+01,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.000000e+00,2.100000e+01,4.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,2.000000e+00,3.400000e+01,5.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
max,4.000000e+00,4.900000e+01,1.400000e+01,7.000000e+00,7.000000e+00,6.000000e+00,1.000000e+00


### Create Useful Cricket Features

#### Create Ball ID  (over 10 ball 3 → 103)

In [58]:
df["ball_id"] = df["over"]*10 + df["ball"]
df.head(3)

,match_id,match_date,inning,over,ball,batter,bowler,non_striker,runs_batter,runs_total,extras,wicket,player_out,wicket_kind,fielder,team,ball_id
0,1000887,2017-01-13,1,0,1,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia,1
1,1000887,2017-01-13,1,0,2,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia,2
2,1000887,2017-01-13,1,0,3,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia,3


#### Identify Boundary Runs

In [59]:
df["is_four"] = (df["runs_batter"] == 4).astype(int)
df["is_six"] = (df["runs_batter"] == 6).astype(int)
df.head(3)

,match_id,match_date,inning,over,ball,batter,bowler,non_striker,runs_batter,runs_total,extras,wicket,player_out,wicket_kind,fielder,team,ball_id,is_four,is_six
0,1000887,2017-01-13,1,0,1,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia,1,0,0
1,1000887,2017-01-13,1,0,2,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia,2,0,0
2,1000887,2017-01-13,1,0,3,DA Warner,Mohammad Amir,TM Head,0,0,0,0,None,None,None,Australia,3,0,0


#### Dot Ball Feature

In [60]:
df["dot_ball"] = (df["runs_total"] == 0).astype(int)
df.tail(3)

,match_id,match_date,inning,over,ball,batter,bowler,non_striker,runs_batter,runs_total,extras,wicket,player_out,wicket_kind,fielder,team,ball_id,is_four,is_six,dot_ball
1639274,997995,2016-08-16,2,47,2,RD Berrington,Rohan Mustafa,PL Mommsen,0,0,0,0,None,None,None,Scotland,472,0,0,1
1639275,997995,2016-08-16,2,47,3,RD Berrington,Rohan Mustafa,PL Mommsen,0,0,0,0,None,None,None,Scotland,473,0,0,1
1639276,997995,2016-08-16,2,47,4,RD Berrington,Rohan Mustafa,PL Mommsen,4,4,0,0,None,None,None,Scotland,474,1,0,0


#### Phase of Match
| Phase     | Overs |
| --------- | ----- |
| Powerplay | 0–10  |
| Middle    | 11–40 |
| Death     | 41–50 |


In [61]:
def match_phase(over):

    if over < 10:
        return "Powerplay"
    elif over < 40:
        return "Middle"
    else:
        return "Death"


df["phase"] = df["over"].apply(match_phase)
df.head(3)

,match_id,match_date,inning,over,ball,batter,bowler,non_striker,runs_batter,runs_total,...,wicket,player_out,wicket_kind,fielder,team,ball_id,is_four,is_six,dot_ball,phase
0,1000887,2017-01-13,1,0,1,DA Warner,Mohammad Amir,TM Head,0,0,...,0,None,None,None,Australia,1,0,0,1,Powerplay
1,1000887,2017-01-13,1,0,2,DA Warner,Mohammad Amir,TM Head,0,0,...,0,None,None,None,Australia,2,0,0,1,Powerplay
2,1000887,2017-01-13,1,0,3,DA Warner,Mohammad Amir,TM Head,0,0,...,0,None,None,None,Australia,3,0,0,1,Powerplay


#### Save Clean Dataset

In [62]:
df.to_csv("../data/odi_ball_by_ball.csv", index=False)

In [63]:
df = pd.read_csv("../data/odi_ball_by_ball.csv")

In [64]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1639277 entries, 0 to 1639276
Data columns (total 21 columns):
 #   Column       Non-Null Count    Dtype 
---  ------       --------------    ----- 
 0   match_id     1639277 non-null  int64 
 1   match_date   1639277 non-null  object
 2   inning       1639277 non-null  int64 
 3   over         1639277 non-null  int64 
 4   ball         1639277 non-null  int64 
 5   batter       1639277 non-null  object
 6   bowler       1639277 non-null  object
 7   non_striker  1639277 non-null  object
 8   runs_batter  1639277 non-null  int64 
 9   runs_total   1639277 non-null  int64 
 10  extras       1639277 non-null  int64 
 11  wicket       1639277 non-null  int64 
 12  player_out   44877 non-null    object
 13  wicket_kind  44877 non-null    object
 14  fielder      28932 non-null    object
 15  team         1639277 non-null  object
 16  ball_id      1639277 non-null  int64 
 17  is_four      1639277 non-null  int64 
 18  is_six       1639277 n

In [65]:
df.isnull().sum()

match_id             0
match_date           0
inning               0
over                 0
ball                 0
batter               0
bowler               0
non_striker          0
runs_batter          0
runs_total           0
extras               0
wicket               0
player_out     1594400
wicket_kind    1594400
fielder        1610345
team                 0
ball_id              0
is_four              0
is_six               0
dot_ball             0
phase                0
dtype: int64